In [1]:
import mitsuba as mi
import drjit as dr
import torch
import torch.nn as nn
mi.set_variant('cuda_ad_rgb')
from tqdm import tqdm
torch.__version__

'2.8.0+cu129'

In [2]:
class TorchTexture(mi.Texture):
    def __init__(self, props: mi.Properties) -> None:
        mi.Texture.__init__(self, props)
        self.network = None        

    def traverse(self, callback):
        if self.network is not None:
            self.network.traverse(callback)
        callback.put("texture", self, mi.ParamFlags.NonDifferentiable)

    def eval(self, si, active=True, dirs=None, norms=None, albedo=None):
        return self.network.eval(si, dirs, norms, albedo)

    def eval_1(self, si, active=True):
        raise NotImplementedError()

    def eval_1_grad(self, *args, **kwargs):
        raise NotImplementedError()

    def eval_3(self, *args, **kwargs):
        raise NotImplementedError()

    def mean(self, *args, **kwargs):
        raise NotImplementedError()

    def to_string(self):
        return (
            "TorchTexture[\n"
            f"  network={self.network}\n"
            "]"
        )

mi.register_texture('torch_texture', TorchTexture)

In [3]:
class ColorMLP(nn.Module):
    def __init__(self, width: int, hidden: int):
        super().__init__()

        in_size = 3
        out_size = 3

        hidden_layers = []
        for _ in range(hidden):
            hidden_layers.append(nn.Linear(width, width))
            hidden_layers.append(nn.LeakyReLU(inplace=True))

        self.network = nn.Sequential(
            nn.Linear(in_size, width),
            nn.LeakyReLU(inplace=True),
            *hidden_layers,
            nn.Linear(width, out_size),
            nn.Sigmoid()
        )

    def forward(self, pts):
        out = self.network(pts)
        return out

In [4]:
def vec_to_tens_safe(vec):
    # A utility function that converts a Vector3f to a TensorXf safely in mitsuba while keeping the gradients;
    # a regular type cast mi.TensorXf(vector) detaches the gradients
    return mi.TensorXf(dr.ravel(vec), shape=(dr.shape(vec)[1], dr.shape(vec)[0]))


In [5]:
class MitsubaWrapper(nn.Module):
    def __init__(self, name: str = None):
        super().__init__()
        self.grad_activator = mi.Vector3f(0)
        self.name = name or type(self).__name__

    def eval(self, si, dirs=None, norms=None, albedo=None):
        result = self._eval(si, dirs, norms, albedo)
        return result

    def traverse(self, callback):
        callback.put("grad_activator", self.grad_activator, mi.ParamFlags.Differentiable)
        self._traverse(callback)

    def _eval(self, pts, dirs, norms, albedo):
        raise NotImplementedError()

    def _traverse(self, callback):
        pass

In [6]:
class MitsubaColorMLPWrapper(MitsubaWrapper):
    def __init__(self, width: int, hidden: int):
        super().__init__("bsdf_color_net")
        self.network = ColorMLP(width, hidden)

    def _eval(self, si, dirs, norms, albedo):
        pts = si.p
        p_tensor = vec_to_tens_safe(pts + self.grad_activator)
        torch_out = self.eval_torch(p_tensor)
        output = dr.unravel(mi.Vector3f, torch_out.array)
        return dr.clip(output, 0, 1)
    
    @dr.wrap(source="drjit", target="torch")
    def eval_torch(self, pts):
        return self.network(pts)
    
    def _traverse(self, callback):
        callback.put("network", self.network, mi.ParamFlags.Differentiable)

In [7]:
scene = mi.load_dict(mi.cornell_box())
gt = mi.render(scene, spp= 512)

In [8]:
# utility function that non-differentiably renders images using Mitsuba aov integrator
def render_nondiff():
    integrator = mi.load_dict({'type': 'aov',
                                'aovs': 'ab:albedo',
                                'my_image': {'type': 'path'}
                                })
    with dr.suspend_grad():
        with torch.no_grad():
            return mi.render(scene, integrator = integrator, spp = 32)

In [9]:
network = MitsubaColorMLPWrapper(256, 2)
network = network.cuda()

In [10]:
texture = mi.load_dict({'type':'torch_texture'})
texture.network = network
bsdf = mi.load_dict({'type':'diffuse', 'reflectance': texture})


In [11]:
objects = ['light', 'floor', 'ceiling', 'back', 'green-wall', 'red-wall', 'small-box', 'large-box']
new_cbox_dict = mi.cornell_box()
for key in objects:
    new_cbox_dict[key]['bsdf'] = bsdf


In [12]:
scene = mi.load_dict(new_cbox_dict)

In [13]:
def mega_kernel(state):
    dr.set_flag(dr.JitFlag.LoopRecord, state)
    dr.set_flag(dr.JitFlag.VCallRecord, state)
    dr.set_flag(dr.JitFlag.VCallOptimize, state)

mega_kernel(False)

In [14]:
img = render_nondiff()

# Albedo map

mi.Bitmap(img[:,:,-7:-4])


Bitmap[
  pixel_format = ya,
  component_format = float32,
  size = [256, 256],
  srgb_gamma = 0,
  struct = Struct<8>[
    float32 Y; // @0, premultiplied alpha
    float32 A; // @4, alpha
  ],
  data = [ 512 KiB of image data ]
]

In [15]:
# Rendering

mi.Bitmap(img[:,:,:4])

Bitmap[
  pixel_format = rgba,
  component_format = float32,
  size = [256, 256],
  srgb_gamma = 0,
  struct = Struct<16>[
    float32 R; // @0, premultiplied alpha
    float32 G; // @4, premultiplied alpha
    float32 B; // @8, premultiplied alpha
    float32 A; // @12, alpha
  ],
  data = [ 1.02e+03 KiB of image data ]
]

In [16]:
params = mi.traverse(texture)
dr.enable_grad(params['grad_activator'])
params

SceneParameters[
  ---------------------------------------------------------------------
  Name              Flags    Type     Parent
  ---------------------------------------------------------------------
  grad_activator    ∂        Vector3f Texture
  network           ∂        ColorMLP Texture
]

In [17]:
optim = torch.optim.Adam(network.parameters(), lr=0.005)

In [29]:
for i in tqdm(range(200)):
    optim.zero_grad()
    img = mi.render(scene, spp = 8, params=params)
    assert dr.grad_enabled(img)
    loss = dr.mean((img-gt)**2)
    dr.backward(loss)
    optim.step()

    dr.flush_malloc_cache()
    torch.cuda.empty_cache()


100%|██████████| 200/200 [01:13<00:00,  2.72it/s]


In [26]:
img = render_nondiff()

# Albedo map
mi.Bitmap(img[:,:,-7:-4])


Bitmap[
  pixel_format = ya,
  component_format = float32,
  size = [256, 256],
  srgb_gamma = 0,
  struct = Struct<8>[
    float32 Y; // @0, premultiplied alpha
    float32 A; // @4, alpha
  ],
  data = [ 512 KiB of image data ]
]

In [27]:
# Rendering

mi.Bitmap(img[:,:,:4])

Bitmap[
  pixel_format = rgba,
  component_format = float32,
  size = [256, 256],
  srgb_gamma = 0,
  struct = Struct<16>[
    float32 R; // @0, premultiplied alpha
    float32 G; // @4, premultiplied alpha
    float32 B; // @8, premultiplied alpha
    float32 A; // @12, alpha
  ],
  data = [ 1.02e+03 KiB of image data ]
]

In [25]:
# Ground truth

mi.Bitmap(gt)

Bitmap[
  pixel_format = rgb,
  component_format = float32,
  size = [256, 256],
  srgb_gamma = 0,
  struct = Struct<12>[
    float32 R; // @0, premultiplied alpha
    float32 G; // @4, premultiplied alpha
    float32 B; // @8, premultiplied alpha
  ],
  data = [ 768 KiB of image data ]
]